# Anotación TE con MMseqs2 (GPU) · Hongos del muestreo

**Autor**: Cristian Osorio · cristiancamiloosoriogranada@gmail.com  
**Objetivo**: anotar los 117 hongos del dataset con MMseqs2 contra la biblioteca PanTEon Fungi.

**Por qué MMseqs2 y no RepeatMasker**:
- RepeatMasker local tarda ~1 h por hongo con la lib Fungi completa (71 MB · 19,797 consensos).
- 117 hongos × 1 h = **5 días** en PC local.
- MMseqs2 con GPU: ~1-2 min por hongo → **~2-4 horas total** para los 117.

## Pasos
1. Instalar MMseqs2
2. Montar Drive (donde subes la lib PanTEon y el manifest)
3. Descargar genomas de hongos desde NCBI
4. Correr búsqueda MMseqs2 con GPU
5. Convertir resultados a formato BED
6. Guardar en Drive para descargar después

## Antes de correr
**En Colab**: Runtime → Change runtime type → GPU (T4 es suficiente).

**En tu Drive**, crear carpeta `InpactorRM/` y subir:
- `panteon_fungi.fasta` (71 MB) — biblioteca de hongos
- `species_download_manifest.tsv` — manifest con URLs de genomas

## 1. Verificar GPU disponible

In [ ]:
!nvidia-smi

## 2. Instalar MMseqs2 (con soporte GPU)

In [ ]:
# MMseqs2 release estable con soporte avx2 + GPU vía CUDA
!wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz
!tar xzf mmseqs-linux-avx2.tar.gz
!cp mmseqs/bin/mmseqs /usr/local/bin/
!mmseqs version

## 3. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/InpactorRM/results_hongos

## 4. Cargar lib PanTEon y manifest desde Drive

In [ ]:
import shutil

# Copia local para mayor velocidad
shutil.copy('/content/drive/MyDrive/InpactorRM/panteon_fungi.fasta',
            '/content/panteon_fungi.fasta')
shutil.copy('/content/drive/MyDrive/InpactorRM/species_download_manifest.tsv',
            '/content/species_download_manifest.tsv')

!ls -lh /content/panteon_fungi.fasta /content/species_download_manifest.tsv
!grep -c "^>" /content/panteon_fungi.fasta

## 5. Construir base MMseqs2 de la biblioteca PanTEon

In [ ]:
!mkdir -p /content/mmseqs_db
!mmseqs createdb /content/panteon_fungi.fasta /content/mmseqs_db/panteon_fungi
!mmseqs createindex /content/mmseqs_db/panteon_fungi /content/mmseqs_tmp --search-type 3
!ls -lh /content/mmseqs_db/

## 6. Procesar los 117 hongos (loop)

Para cada hongo:
- Descarga de NCBI
- MMseqs2 search
- Convierte a BED
- Guarda en Drive

In [ ]:
import csv, os, subprocess, time
from pathlib import Path

MANIFEST = '/content/species_download_manifest.tsv'
OUT_DRIVE = '/content/drive/MyDrive/InpactorRM/results_hongos'
DB = '/content/mmseqs_db/panteon_fungi'

# Leer manifest y filtrar Hongos
hongos = []
with open(MANIFEST) as f:
    for row in csv.DictReader(f, delimiter='\t'):
        if row['reino'] == 'Hongos' and row['status'] == 'found':
            hongos.append(row)

print(f'Hongos a procesar: {len(hongos)}')

In [ ]:
def process_hongo(row, idx):
    especie = row['especie']
    acc = row['assembly_accession']
    url = row['url_fna'].strip()
    split = row['split']

    sp_safe = ''.join(c if c.isalnum() or c in '_-' else '_' for c in especie.replace(' ', '_'))
    tag = f'{sp_safe}__{acc}'

    out_bed = f'{OUT_DRIVE}/{split}/{tag}.bed'
    os.makedirs(f'{OUT_DRIVE}/{split}', exist_ok=True)
    if os.path.exists(out_bed):
        return 'SKIP', 0

    t0 = time.time()

    # 1. Descargar genoma
    genome = f'/content/tmp_genome.fna.gz'
    r = subprocess.run(['wget', '-q', '--timeout=600', '-O', genome, url])
    if r.returncode != 0 or not os.path.getsize(genome):
        return 'DL_FAIL', time.time() - t0

    # 2. Descomprimir
    subprocess.run(['gunzip', '-f', genome])
    genome_fasta = '/content/tmp_genome.fna'

    # 3. MMseqs2 createdb
    subprocess.run(['mmseqs', 'createdb', genome_fasta, '/content/query_db'],
                   capture_output=True)

    # 4. Search (con GPU si disponible)
    subprocess.run(['mmseqs', 'search',
                    '/content/query_db', DB,
                    '/content/result_db', '/content/mmseqs_tmp',
                    '--search-type', '3',
                    '-s', '7.5',
                    '--min-seq-id', '0.6',
                    '-c', '0.5',
                    '--threads', '4'],
                   capture_output=True)

    # 5. Convertir a formato tabular
    subprocess.run(['mmseqs', 'convertalis',
                    '/content/query_db', DB,
                    '/content/result_db', '/content/result.m8',
                    '--format-output', 'query,target,pident,alnlen,qstart,qend,evalue'],
                   capture_output=True)

    # 6. Convertir m8 a BED
    with open('/content/result.m8') as fi, open(out_bed, 'w') as fo:
        for line in fi:
            s = line.strip().split('\t')
            if len(s) < 7: continue
            chrom, target, pident, alnlen, qstart, qend, evalue = s[:7]
            start = min(int(qstart), int(qend)) - 1
            end = max(int(qstart), int(qend))
            # El target tiene la forma PDB00000001#CLASSI/LTR/GYPSY
            tokens = target.split('#')
            te_id = tokens[0]
            cls = tokens[1] if len(tokens) > 1 else 'Unknown'
            fo.write(f'{chrom}\t{start}\t{end}\t{te_id}|{cls}\t{pident}\t.\n')

    # Cleanup
    for f in ['/content/tmp_genome.fna', '/content/query_db*',
              '/content/result_db*', '/content/result.m8']:
        subprocess.run(['bash', '-c', f'rm -rf {f}'])

    return 'OK', time.time() - t0

In [ ]:
# Loop principal con progreso visible
import time
n_ok = n_skip = n_fail = 0
T_INIT = time.time()

for i, row in enumerate(hongos, 1):
    status, dt = process_hongo(row, i)
    if status == 'OK': n_ok += 1
    elif status == 'SKIP': n_skip += 1
    else: n_fail += 1
    elapsed = time.time() - T_INIT
    eta = (elapsed / i) * (len(hongos) - i) if i > 0 else 0
    print(f'[{i:3d}/{len(hongos)}] {status:7s} {row["especie"][:40]:40s} '
          f'({dt:5.1f}s) · OK={n_ok} FAIL={n_fail} · ETA {eta/60:.0f} min')

print(f'\nCompletado: {n_ok} OK · {n_skip} SKIP · {n_fail} FAIL')
print(f'Tiempo total: {(time.time() - T_INIT)/60:.1f} min')

## 7. Verificar resultados en Drive

In [ ]:
!find /content/drive/MyDrive/InpactorRM/results_hongos -name '*.bed' | wc -l
!du -sh /content/drive/MyDrive/InpactorRM/results_hongos

## 8. Después (en tu PC local)

Para bajar los BEDs resultantes:

```bash
# Si tienes rclone configurado con tu Drive:
rclone sync drive:InpactorRM/results_hongos /path/local/

# O descarga el folder como ZIP desde la UI de Drive
```